# Advanced RAG — Re-ranking, Hybrid Search, Query Expansion

Basic vector RAG retrieves by embedding similarity alone. That fails when:

- Users say **"money back"** but policies say **"refund"**
- Exact IDs (`SHOPCO*`, `ORD-1001`) need **keyword** precision
- The best chunk ranks **#4** in vector search but **#1** for the full question

Three upgrades fix most production retrieval gaps:

| Technique | What it does | Fixes |
|-----------|--------------|-------|
| **Query expansion** | Add synonyms / multi-query variants | Vocabulary mismatch |
| **Hybrid search** | Fuse vector + keyword (BM25) ranks | IDs, rare terms, exact phrases |
| **Re-ranking** | Rescore top-N with a stronger model | Precision on final context window |

This notebook implements all three on **ShopCo Policy RAG** (TF-IDF + FAISS), compares baseline vs advanced retrieval on hard queries, and wires the stack into a support agent workflow.

Everything is self-contained plain Python. Optional cells use OpenAI for query expansion or embeddings.

In [ ]:
"""
ShopCo Advanced RAG — hybrid search, expansion, re-ranking lab.
"""

import json
import math
import os
import re
import uuid
from collections import Counter
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any

import faiss
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)

USE_LIVE_LLM = False


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


POLICY_DOCS: list[dict[str, str]] = [
    {
        "id": "pol-returns-01",
        "title": "Returns Policy",
        "text": "Customers may return items within 30 days of delivery. Items must be unused and in original packaging. Refunds are processed within 5-7 business days. Final-sale items cannot be returned.",
    },
    {
        "id": "pol-shipping-02",
        "title": "Shipping Policy",
        "text": "Standard shipping is free on orders over $50. Express shipping at checkout. Tracking numbers are emailed when the carrier scans the package.",
    },
    {
        "id": "pol-billing-03",
        "title": "Billing Policy",
        "text": "Charges appear as SHOPCO* on credit card statements. Partial refunds may take up to two billing cycles to appear. Disputes require order ID.",
    },
    {
        "id": "faq-tracking-04",
        "title": "Tracking FAQ",
        "text": "Shipped orders have tracking numbers in the confirmation email. Processing orders do not yet have tracking. Use the carrier portal once tracking is assigned.",
    },
    {
        "id": "faq-warranty-05",
        "title": "Warranty FAQ",
        "text": "Electronics include a 1-year manufacturer warranty. Claims require proof of purchase and serial number. Warranty does not cover accidental damage.",
    },
]

CHUNKS: list[dict[str, str]] = [
    {"chunk_id": d["id"], "doc_id": d["id"], "title": d["title"], "text": d["text"]}
    for d in POLICY_DOCS
]

print(f"Indexed corpus: {len(CHUNKS)} chunks")

---

## 1. Baseline Pipeline — Vector-Only FAISS

The starting point: embed chunks with TF-IDF, L2-normalize, search with `IndexFlatIP` (cosine similarity).

In [ ]:
@dataclass
class RetrievalHit:
    chunk_id: str
    doc_id: str
    title: str
    text: str
    vector_score: float = 0.0
    keyword_score: float = 0.0
    fusion_score: float = 0.0
    rerank_score: float = 0.0
    source_query: str = ""


class VectorIndex:
    def __init__(self, chunks: list[dict[str, str]]) -> None:
        self.chunks = chunks
        self._chunk_by_id = {c["chunk_id"]: c for c in chunks}
        texts = [f"{c['title']} {c['text']}" for c in chunks]
        self.vectorizer = TfidfVectorizer(stop_words="english")
        matrix = self.vectorizer.fit_transform(texts).astype(np.float32)
        self.vectors = matrix.toarray()
        faiss.normalize_L2(self.vectors)
        self.index = faiss.IndexFlatIP(self.vectors.shape[1])
        self.index.add(self.vectors)

    def search_vector(self, query: str, top_k: int = 5) -> list[RetrievalHit]:
        q = self.vectorizer.transform([query]).astype(np.float32).toarray()
        faiss.normalize_L2(q)
        scores, indices = self.index.search(q, top_k)
        hits: list[RetrievalHit] = []
        for score, idx in zip(scores[0], indices[0]):
            if idx < 0:
                continue
            c = self.chunks[idx]
            hits.append(RetrievalHit(
                chunk_id=c["chunk_id"], doc_id=c["doc_id"],
                title=c["title"], text=c["text"],
                vector_score=float(score), source_query=query,
            ))
        return hits


vector_index = VectorIndex(CHUNKS)
baseline = vector_index.search_vector("return policy", top_k=3)
print("Vector-only:", [(h.chunk_id, round(h.vector_score, 3)) for h in baseline])

---

## 2. Keyword Channel — BM25-Style Scoring

Hybrid search adds a **sparse** channel for exact tokens (`SHOPCO*`, `30 days`, order IDs). We implement a lightweight BM25-like scorer.

In [ ]:
def tokenize(text: str) -> list[str]:
    return [t for t in re.split(r"\W+", text.lower()) if len(t) > 1]


class KeywordIndex:
    def __init__(self, chunks: list[dict[str, str]], k1: float = 1.5, b: float = 0.75) -> None:
        self.chunks = chunks
        self.k1, self.b = k1, b
        self.doc_tokens = [tokenize(f"{c['title']} {c['text']}") for c in chunks]
        self.avg_dl = sum(len(d) for d in self.doc_tokens) / len(chunks)
        self.df: Counter[str] = Counter()
        for toks in self.doc_tokens:
            for t in set(toks):
                self.df[t] += 1
        self.n_docs = len(chunks)

    def _idf(self, term: str) -> float:
        df = self.df.get(term, 0)
        return math.log(1 + (self.n_docs - df + 0.5) / (df + 0.5))

    def score(self, query: str, doc_idx: int) -> float:
        q_terms = tokenize(query)
        doc = self.doc_tokens[doc_idx]
        dl = len(doc)
        tf_counter = Counter(doc)
        total = 0.0
        for term in q_terms:
            tf = tf_counter.get(term, 0)
            if tf == 0:
                continue
            idf = self._idf(term)
            denom = tf + self.k1 * (1 - self.b + self.b * dl / self.avg_dl)
            total += idf * (tf * (self.k1 + 1)) / denom
        return total

    def search_keyword(self, query: str, top_k: int = 5) -> list[RetrievalHit]:
        scored = [(self.score(query, i), i) for i in range(len(self.chunks))]
        scored.sort(key=lambda x: -x[0])
        hits: list[RetrievalHit] = []
        for score, idx in scored[:top_k]:
            c = self.chunks[idx]
            hits.append(RetrievalHit(
                chunk_id=c["chunk_id"], doc_id=c["doc_id"],
                title=c["title"], text=c["text"],
                keyword_score=float(score), source_query=query,
            ))
        return hits


keyword_index = KeywordIndex(CHUNKS)
kw = keyword_index.search_keyword("SHOPCO* statement charge", top_k=3)
print("Keyword-only:", [(h.chunk_id, round(h.keyword_score, 3)) for h in kw])

---

## 3. Where Baseline Retrieval Fails

Hard queries expose vocabulary mismatch and exact-token needs.

In [ ]:
HARD_QUERIES = [
    {"q": "I want my money back", "expect": "pol-returns-01"},
    {"q": "SHOPCO* charge on my statement", "expect": "pol-billing-03"},
    {"q": "Where is my package", "expect": "faq-tracking-04"},
    {"q": "free delivery threshold", "expect": "pol-shipping-02"},
]


def top1_doc_id(hits: list[RetrievalHit]) -> str:
    return hits[0].chunk_id if hits else ""


print(f"{'Query':<35} {'Vector top-1':<22} {'Keyword top-1':<22} Expected")
print("-" * 95)
for row in HARD_QUERIES:
    v = top1_doc_id(vector_index.search_vector(row["q"]))
    k = top1_doc_id(keyword_index.search_keyword(row["q"]))
    ok_v = "✓" if v == row["expect"] else "✗"
    ok_k = "✓" if k == row["expect"] else "✗"
    print(f"{row['q']:<35} {v:<20}{ok_v}  {k:<20}{ok_k}  {row['expect']}")

---

## 4. Query Expansion

**Query expansion** rewrites the user question before search:

1. **Synonym map** — `money back` → `refund`
2. **Multi-query** — generate 2–3 variants, retrieve each, merge results
3. **LLM rewrite** (optional) — paraphrase for policy vocabulary

Expansion increases recall; hybrid fusion and re-ranking restore precision.

In [ ]:
SYNONYM_MAP: dict[str, list[str]] = {
    "money back": ["refund", "return", "reimbursement"],
    "package": ["tracking", "shipment", "delivery"],
    "charge": ["billing", "statement", "SHOPCO*"],
    "free delivery": ["free shipping", "shipping over $50"],
    "warranty claim": ["manufacturer warranty", "proof of purchase"],
}


def expand_query_rule_based(query: str) -> list[str]:
    variants = [query]
    q_lower = query.lower()
    for phrase, additions in SYNONYM_MAP.items():
        if phrase in q_lower:
            for add in additions:
                variants.append(query + " " + add)
    # de-duplicate preserving order
    seen: set[str] = set()
    out: list[str] = []
    for v in variants:
        if v not in seen:
            seen.add(v)
            out.append(v)
    return out


def expand_query_live_llm(query: str) -> list[str]:
    from langchain_core.messages import HumanMessage, SystemMessage
    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)
    resp = llm.invoke([
        SystemMessage(content="Generate 3 search queries for ShopCo policy retrieval. Return JSON list of strings."),
        HumanMessage(content=query),
    ])
    try:
        variants = json.loads(str(resp.content))
        return [query] + [v for v in variants if isinstance(v, str)]
    except json.JSONDecodeError:
        return expand_query_rule_based(query)


def expand_query(query: str) -> list[str]:
    if USE_LIVE_LLM:
        return expand_query_live_llm(query)
    return expand_query_rule_based(query)


for q in ["I want my money back", "Where is my package"]:
    print(f"Q: {q}")
    print(f"  variants: {expand_query(q)}")

---

## 5. Hybrid Search — Reciprocal Rank Fusion (RRF)

Run **vector** and **keyword** channels independently, then fuse ranked lists with **RRF**:

$$\text{RRF}(d) = \sum_{r \in \text{rankings}} \frac{1}{k + \text{rank}_r(d)}$$

Default `k=60` dampens dominance of any single channel.

In [ ]:
def reciprocal_rank_fusion(
    ranked_lists: list[list[RetrievalHit]],
    k: int = 60,
) -> list[RetrievalHit]:
    scores: dict[str, float] = {}
    meta: dict[str, RetrievalHit] = {}
    for ranked in ranked_lists:
        for rank, hit in enumerate(ranked, start=1):
            scores[hit.chunk_id] = scores.get(hit.chunk_id, 0.0) + 1.0 / (k + rank)
            if hit.chunk_id not in meta:
                meta[hit.chunk_id] = hit
            else:
                meta[hit.chunk_id].vector_score = max(meta[hit.chunk_id].vector_score, hit.vector_score)
                meta[hit.chunk_id].keyword_score = max(meta[hit.chunk_id].keyword_score, hit.keyword_score)
    fused = sorted(scores.items(), key=lambda x: -x[1])
    out: list[RetrievalHit] = []
    for chunk_id, fusion in fused:
        h = meta[chunk_id]
        h.fusion_score = fusion
        out.append(h)
    return out


def hybrid_search(query: str, top_k: int = 5) -> list[RetrievalHit]:
    vec = vector_index.search_vector(query, top_k=top_k * 2)
    kw = keyword_index.search_keyword(query, top_k=top_k * 2)
    return reciprocal_rank_fusion([vec, kw])[:top_k]


hyb = hybrid_search("SHOPCO* charge on my statement")
print("Hybrid top-3:", [(h.chunk_id, round(h.fusion_score, 4)) for h in hyb[:3]])

---

## 6. Re-Ranking — Precision on Top Candidates

Retrieve a **wide** candidate set (e.g. top-20), then **re-rank** with a stronger scorer before passing top-3 to the agent.

Production uses cross-encoder models (`ms-marco-MiniLM`). Here we use an interpretable feature-based reranker:

In [ ]:
def title_overlap(query: str, title: str) -> float:
    q_terms = set(tokenize(query))
    t_terms = set(tokenize(title))
    if not q_terms:
        return 0.0
    return len(q_terms & t_terms) / len(q_terms)


def exact_phrase_bonus(query: str, text: str) -> float:
    bonus = 0.0
    for token in ("SHOPCO*", "30 days", "5-7 business days", "$50"):
        if token.lower() in query.lower() and token.lower() in text.lower():
            bonus += 0.15
    return min(bonus, 0.45)


def rerank_hits(query: str, candidates: list[RetrievalHit], top_k: int = 3) -> list[RetrievalHit]:
    for hit in candidates:
        hit.rerank_score = (
            0.35 * hit.fusion_score
            + 0.25 * hit.vector_score
            + 0.20 * (hit.keyword_score / (1 + hit.keyword_score))
            + 0.12 * title_overlap(query, hit.title)
            + 0.08 * exact_phrase_bonus(query, hit.text)
        )
    ranked = sorted(candidates, key=lambda h: -h.rerank_score)
    return ranked[:top_k]


candidates = hybrid_search("I want my money back", top_k=10)
reranked = rerank_hits("I want my money back", candidates)
print("Re-ranked:", [(h.chunk_id, round(h.rerank_score, 3)) for h in reranked])

---

## 7. Advanced Retriever — Full Stack

Combine **expansion → multi-query hybrid → re-rank** in one class for agent tools.

In [ ]:
@dataclass
class AdvancedRetrievalResult:
    query: str
    expanded_queries: list[str]
    hits: list[RetrievalHit]
    latency_ms_simulated: int
    trace: list[str] = field(default_factory=list)


class AdvancedRetriever:
    def __init__(self, vector_idx: VectorIndex, keyword_idx: KeywordIndex) -> None:
        self.vector_idx = vector_idx
        self.keyword_idx = keyword_idx

    def retrieve(self, query: str, final_k: int = 3, candidate_k: int = 10) -> AdvancedRetrievalResult:
        trace: list[str] = []
        variants = expand_query(query)
        trace.append(f"expand:{len(variants)} variants")

        all_ranked: list[list[RetrievalHit]] = []
        for v in variants:
            vec = self.vector_idx.search_vector(v, top_k=candidate_k)
            kw = self.keyword_idx.search_keyword(v, top_k=candidate_k)
            all_ranked.append(reciprocal_rank_fusion([vec, kw]))
        trace.append(f"hybrid:{len(all_ranked)} channel runs")

        fused = reciprocal_rank_fusion(all_ranked)[:candidate_k]
        final = rerank_hits(query, fused, top_k=final_k)
        trace.append(f"rerank:top{final_k}")

        return AdvancedRetrievalResult(
            query=query,
            expanded_queries=variants,
            hits=final,
            latency_ms_simulated=35 + 15 * len(variants),
            trace=trace,
        )


advanced = AdvancedRetriever(vector_index, keyword_index)
result = advanced.retrieve("I want my money back")
print("Trace:", result.trace)
print("Top hit:", result.hits[0].chunk_id, result.hits[0].text[:60], "...")

---

## 8. Agent Tool — `search_policy_advanced`

In [ ]:
def search_policy_advanced(query: str, top_k: int = 3) -> dict[str, Any]:
    """Agent-callable tool returning structured evidence for grounding."""
    r = advanced.retrieve(query, final_k=top_k)
    return {
        "query": r.query,
        "expanded_queries": r.expanded_queries,
        "hits": [
            {
                "chunk_id": h.chunk_id,
                "title": h.title,
                "text": h.text,
                "scores": {
                    "vector": round(h.vector_score, 4),
                    "keyword": round(h.keyword_score, 4),
                    "fusion": round(h.fusion_score, 4),
                    "rerank": round(h.rerank_score, 4),
                },
            }
            for h in r.hits
        ],
        "trace": r.trace,
    }


tool_out = search_policy_advanced("SHOPCO* charge on my statement")
print(pretty(tool_out))

---

## 9. Support Agent Workflow with Advanced Retrieval

In [ ]:
@dataclass
class AgentState:
    ticket_id: str
    question: str
    retrieval: dict[str, Any] | None = None
    answer: str = ""
    trace: list[str] = field(default_factory=list)


def compose_from_hits(question: str, hits: list[dict[str, Any]]) -> str:
    if not hits:
        return "I could not find a verified ShopCo policy for that question."
    top = hits[0]
    cite = top["chunk_id"]
    return f"Based on [{cite}] {top['title']}: {top['text'][:120]}..."


def run_support_agent(ticket_id: str, question: str) -> AgentState:
    state = AgentState(ticket_id=ticket_id, question=question)
    state.retrieval = search_policy_advanced(question)
    state.trace.append("tool:search_policy_advanced")
    state.answer = compose_from_hits(question, state.retrieval["hits"])
    state.trace.append("compose:grounded")
    return state


for q in ["I want my money back", "free delivery threshold"]:
    s = run_support_agent(str(uuid.uuid4())[:8], q)
    print(f"Q: {q}")
    print(f"  {s.answer[:90]}...\n")

---

## 10. Strategy Comparison on Hard Queries

In [ ]:
def eval_strategy(search_fn, label: str) -> float:
    hits = 0
    for row in HARD_QUERIES:
        if label == "vector":
            got = top1_doc_id(vector_index.search_vector(row["q"]))
        elif label == "keyword":
            got = top1_doc_id(keyword_index.search_keyword(row["q"]))
        elif label == "hybrid":
            got = top1_doc_id(hybrid_search(row["q"]))
        else:
            r = advanced.retrieve(row["q"])
            got = r.hits[0].chunk_id if r.hits else ""
        if got == row["expect"]:
            hits += 1
    return hits / len(HARD_QUERIES)


print("Top-1 accuracy on hard queries:")
for name in ("vector", "keyword", "hybrid", "advanced"):
    print(f"  {name:<12} {eval_strategy(None, name):.0%}")

---

## 11. Latency Budget — Retrieve Wide, Rerank Narrow

| Stage | Typical cost | Agent impact |
|-------|--------------|--------------|
| Query expansion | +1–3 LLM calls OR rule map | Better recall |
| Vector search | ~5–20 ms (FAISS) | Base latency |
| Keyword search | ~1–10 ms | Exact token hits |
| RRF fusion | negligible | Combines channels |
| Re-rank top-20 | +50–200 ms (cross-encoder) | Precision for compose |

**Pattern:** `candidate_k=20` → rerank → `final_k=3` for the LLM context window.

In [ ]:
LATENCY_PROFILE = [
    ("expand (rule)", 5),
    ("vector x3 variants", 30),
    ("keyword x3 variants", 15),
    ("RRF + rerank", 25),
]
total = sum(ms for _, ms in LATENCY_PROFILE)
print(f"Simulated advanced retrieval latency: ~{total} ms")
for stage, ms in LATENCY_PROFILE:
    print(f"  {stage:<25} {ms:>4} ms")

---

## 12. Pipeline Placement in Agents

```
  user question
       │
       ▼
  query expansion ──► multi-query hybrid (FAISS + BM25)
       │                      │
       │                      ▼
       │               RRF fusion (top-20)
       │                      │
       │                      ▼
       │               cross-encoder rerank (top-3)
       │                      │
       └──────────────────────┼──► evidence bundle → compose agent
```

The retriever agent (or `search_policy` tool) owns this stack; composer agents receive only the final hits.

---

## 13. Tuning Knobs

| Knob | Effect | Start value |
|------|--------|-------------|
| `candidate_k` | Recall before rerank | 10–20 |
| `final_k` | Chunks in LLM context | 3–5 |
| RRF `k` | Fusion smoothness | 60 |
| Expansion variants | Recall vs latency | 2–4 |
| Rerank model | Precision | cross-encoder in prod |
| Vector weight in rerank | Semantic vs lexical balance | 0.25–0.40 |

---

## 14. Common Mistakes

| Mistake | Symptom | Fix |
|---------|---------|-----|
| Vector-only on support tickets | Misses `SHOPCO*`, SKUs | Add keyword channel |
| Expansion without rerank | Noisy top-k | RRF + rerank |
| Rerank entire corpus | Timeout | Rerank top-20 only |
| Skip eval on hard queries | Happy path only metrics | `HARD_QUERIES` golden set |
| Same k for retrieve and context | Too many tokens | `candidate_k` >> `final_k` |

---

## 15. Optional Live LLM Query Expansion

Set `USE_LIVE_LLM = True` in the setup cell to expand queries via `gpt-4o-mini` instead of the rule-based synonym map.

In [ ]:
if USE_LIVE_LLM:
    r = AdvancedRetriever(vector_index, keyword_index)
    live = r.retrieve("My card shows a weird SHOPCO charge")
    print("Expanded:", live.expanded_queries)
    print("Top:", live.hits[0].chunk_id)
else:
    print("Offline mode. Advanced retrieval top-1 on 'money back':",
          advanced.retrieve("I want my money back").hits[0].chunk_id)
    print("Set USE_LIVE_LLM = True for LLM query expansion.")

---

## 16. Quiz

1. Why does hybrid search outperform vector-only on `SHOPCO*` queries?
2. What problem does query expansion solve, and what problem does re-ranking solve?
3. Explain RRF in one sentence.
4. Why retrieve `candidate_k=20` but pass only `final_k=3` to the agent?
5. When would you skip expansion and use hybrid search only?

---

## 17. Summary

**Advanced RAG** layers techniques on top of a FAISS vector index:

- **Query expansion** — bridge vocabulary gaps (`money back` → `refund`)
- **Hybrid search** — FAISS + BM25 fused with **RRF**
- **Re-ranking** — rescore top candidates for precision before the compose agent

On ShopCo hard queries, vector-only retrieval fails on paraphrases and exact tokens; the **AdvancedRetriever** stack improves top-1 accuracy by combining all three. Wire it as `search_policy_advanced` in your agent tool loop and measure with a golden set of difficult queries.